# Eurovision C: победитель национального телеголосования

Одно наблюдение — **голосование страны в конкретном году**. Предсказываем,
какой финалист получит от неё 12 зрительских баллов. Это не общий победитель конкурса.

| Модель | Полезность получателя B для голосующего A |
|---|---|
| M0 | Историческая сила получателя $a_B$ |
| M1 | $a_B$ + признаки получателя и пары A → B |
| M2 | Те же признаки + иерархия получателей и направленных пар |

Вероятности внутри каждого допустимого набора суммируются в 1.
**2025 — отдельный тест; он не участвует в обучении или выборе настроек.**
Rest of the World и искусственное телеголосование San Marino исключены;
голосующие страны, не прошедшие в финал, сохраняются.


In [ ]:
%pip -q install "numpyro==0.19.0" "jax[cpu]==0.7.2" "arviz==0.22.0" openpyxl


In [ ]:
from pathlib import Path
import hashlib
import json
import time
import warnings
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter, MaxNLocator
from matplotlib.lines import Line2D
from scipy.special import logsumexp, softmax
from IPython.display import display, Markdown

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.scipy.special import logsumexp as jax_logsumexp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
import arviz as az

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/eurovision_features.xlsx").is_file())
DATA_PATH = ROOT / "data/eurovision_features.xlsx"
import os
EXPERIMENT = os.environ.get("EUROVISION_EXPERIMENT", "market")  # market / matched_core / full_core
OUTPUT_DIR = ROOT / "results/generated/task_c" / EXPERIMENT
HOLDOUT_YEAR = 2025
SEED = 20261025
MODELS = ("M0", "M1", "M2")
TARGET = "target_first_place"
COUNTRY_SD, BETA_SD, TAU_SD, PAIR_SD = 0.5, 0.5, 0.5, 0.5
NUM_CHAINS, NUM_WARMUP, NUM_SAMPLES = 4, 1000, 1000
RUN_SENSITIVITY = True
MODEL_VERSION = "C-country-pair-softmax-v1"
RECIPIENT_CORE = [
    "running_position", "host", "english_only", "english_mixed", "bpm",
    "danceability", "dynamic_complexity", "log_word_count", "repetitiveness",
]
PAIR_FEATURES = ["past_pair_support", "past_pair_count", "log_distance", "shared_border"]
BINARY_FEATURES = {"host", "english_only", "english_mixed", "shared_border"}
EXPERIMENTS = {
    "market": ("C С рынком", RECIPIENT_CORE + ["market_strength"]),
    "matched_core": ("C С рынком", RECIPIENT_CORE),
    "full_core": ("C Национальное голосование", RECIPIENT_CORE),
}
if EXPERIMENT not in EXPERIMENTS:
    raise ValueError(f"Выберите эксперимент из {list(EXPERIMENTS)}")
SHEET, RECIPIENT_FEATURES = EXPERIMENTS[EXPERIMENT]
FEATURES = RECIPIENT_FEATURES + PAIR_FEATURES
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
COLORS = {"M0": "#65758A", "M1": "#3569B8", "M2": "#168B83"}
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11,
                     "figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False, "axes.titleweight": "bold"})
pd.set_option("display.max_rows", 45)
pd.set_option("display.precision", 4)
print(f"NumPyro {numpyro.__version__}; JAX {jax.__version__}; ArviZ {az.__version__}")


## 1. Данные и допустимые кандидаты

По умолчанию включён `market_strength`. `matched_core` исключает котировки
на тех же наблюдениях. Время архивных котировок неизвестно — принятое ограничение.
Проверяем весь набор кандидатов по составу финала, а не только положительные исходы.
Самоголосование запрещено; в каждом голосовании ровно один победитель.
Справочник стран содержит только обозначения из файла: это не обучающие исходы.
У стран и пар без тренировочных наблюдений сохраняется неопределённость из prior.


In [ ]:
if not DATA_PATH.is_file():
    raise FileNotFoundError("Укажите путь к Excel в DATA_PATH.")
raw = pd.read_excel(DATA_PATH, sheet_name=SHEET)
roster = pd.read_excel(DATA_PATH, sheet_name="A Топ-10", usecols=["year", "country"])
required = ["year", "voter", "recipient", "candidate_count", TARGET] + FEATURES
if not set(required) <= set(raw.columns):
    raise ValueError(f"Не хватает столбцов: {sorted(set(required)-set(raw.columns))}")
assert not raw[required].isna().any().any()
assert not raw.duplicated(["year", "voter", "recipient"]).any()
assert raw[TARGET].isin([0, 1]).all()
assert np.isfinite(raw[FEATURES].to_numpy(float)).all()
assert set(raw.year) == set(range(2016, HOLDOUT_YEAR+1)) - {2020}
assert not raw.voter.isin(["SM", "ROW", "RW"]).any()
finalists = roster.groupby("year").country.agg(set).to_dict()
for (year, voter), g in raw.groupby(["year", "voter"]):
    assert set(g.recipient) == finalists[year] - {voter}, (year, voter)
    assert g.candidate_count.eq(len(g)).all()
    assert g[TARGET].sum() == 1
assert raw.groupby(["year", "recipient"])[RECIPIENT_FEATURES].nunique().le(1).all().all()
for col in BINARY_FEATURES:
    assert raw[col].isin([0, 1]).all(), col
COUNTRIES = sorted(set(raw.voter) | set(raw.recipient))
COUNTRY_INDEX = {c: i for i, c in enumerate(COUNTRIES)}
development = raw.loc[raw.year.lt(HOLDOUT_YEAR)].copy()
holdout_design = raw.loc[raw.year.eq(HOLDOUT_YEAR),
                         ["year", "voter", "recipient", "candidate_count"]+FEATURES].copy()
holdout_truth = raw.loc[raw.year.eq(HOLDOUT_YEAR), ["year", "voter", "recipient", TARGET]].copy()
DEV_YEARS = [2021, 2022, 2023, 2024]
assert set(DEV_YEARS) <= set(development.year)
display(raw.groupby("year").agg(candidate_rows=("recipient", "size"), national_votes=("voter", "nunique")))
print("Признаки M1/M2:", ", ".join(FEATURES))


## 2. Подготовка голосований

Характеристики получателя масштабируем по уникальным `(год, получатель)` в train:
многократное повторение одной песни у разных голосующих не увеличивает её вес.
Признаки пар масштабируем по допустимым парам в train. Индикаторы 0/1 сохраняем.
Padding нужен только для массивов одинаковой ширины; он не добавляет кандидатов.


In [ ]:
def pack_votes(frame, X, with_target):
    groups = list(frame.groupby(["year", "voter"], sort=True))
    width = max(len(g) for _, g in groups)
    shape = (len(groups), width)
    packed = dict(X=np.zeros(shape+(len(FEATURES),)), recipient_idx=np.zeros(shape, dtype=int),
                  voter_idx=np.zeros(len(groups), dtype=int), mask=np.zeros(shape, dtype=bool))
    if with_target:
        packed["y"] = np.zeros(len(groups), dtype=int)
    for i, ((year, voter), g) in enumerate(groups):
        n = len(g)
        packed["X"][i, :n] = X[g.index]
        packed["recipient_idx"][i, :n] = g.recipient.map(COUNTRY_INDEX)
        packed["voter_idx"][i] = COUNTRY_INDEX[voter]
        packed["mask"][i, :n] = True
        if with_target:
            assert g[TARGET].sum() == 1
            packed["y"][i] = int(np.flatnonzero(g[TARGET].to_numpy())[0])
    return packed


def prepare_fold(train, design):
    assert train.year.max() < design.year.min() <= HOLDOUT_YEAR
    assert train.year.max() < HOLDOUT_YEAR and TARGET not in design.columns
    train = train.sort_values(["year", "voter", "recipient"]).reset_index(drop=True).copy()
    test = design.sort_values(["year", "voter", "recipient"]).reset_index(drop=True).copy()
    unique_songs = train.drop_duplicates(["year", "recipient"])
    mean, scale = pd.Series(0.0, index=FEATURES), pd.Series(1.0, index=FEATURES)
    for cols, sample in [(RECIPIENT_FEATURES, unique_songs), (PAIR_FEATURES, train)]:
        continuous = [c for c in cols if c not in BINARY_FEATURES]
        mean.loc[continuous] = sample[continuous].mean()
        scale.loc[continuous] = sample[continuous].std(ddof=0).replace(0, 1)
    packed = pack_votes(train, ((train[FEATURES]-mean)/scale).to_numpy(float), True)
    test_packed = pack_votes(test, ((test[FEATURES]-mean)/scale).to_numpy(float), False)
    return packed | dict(train=train, test=test, test_packed=test_packed,
                         mean=mean, scale=scale, countries=COUNTRIES)


## 3. M0, M1 и M2

$$P(W_{At}=B\mid\theta)=softmax_{B\in C_{At}}(\eta_{ABt}).$$
M0/M1: $a_B=0.5(z_B-\bar z)$, независимые $z_B\sim N(0,1)$.
M0 обучает силу получателей без признаков; M1 добавляет $x_{ABt}^\top\beta$.
Для M1/M2 $\beta_j\sim N(0,0.5^2)$ после стандартизации.

M2: масштаб получателей $\tau_a\sim HalfNormal(0.5)$ и остаточный эффект пары
$d_{AB}=\tau_d(Z_{AB}-\bar Z_{A\cdot}-\bar Z_{\cdot B}+\bar Z)$,
$Z_{AB}\sim N(0,1)$, $\tau_d\sim HalfNormal(0.5)$.
Центрируем полную матрицу, затем маскируем запрещённых кандидатов.
У M2 одновременно появляется иерархия и индивидуальная поддержка пар:
сравнение с M1 оценивает их совместный вклад.
Общий intercept и intercept голосующего сокращаются в softmax и не добавляются.


In [ ]:
def centered_pairs(z):
    return z-z.mean(axis=-1, keepdims=True)-z.mean(axis=-2, keepdims=True)+z.mean(axis=(-2, -1), keepdims=True)


def categorical_log_probs(eta, mask):
    safe = jnp.where(mask, eta, -1e30)
    return safe-jax_logsumexp(safe, axis=-1, keepdims=True)


def choice_model(X, recipient_idx, voter_idx, mask, y, n_countries, model_id, prior_multiplier=1.0):
    z = numpyro.sample("z", dist.Normal(0, 1).expand([n_countries]).to_event(1))
    sigma = (numpyro.sample("tau_country", dist.HalfNormal(TAU_SD*prior_multiplier))
             if model_id == "M2" else COUNTRY_SD*prior_multiplier)
    a = numpyro.deterministic("a", sigma*(z-z.mean()))
    eta = a[recipient_idx]
    if model_id != "M0":
        beta = numpyro.sample("beta", dist.Normal(0, BETA_SD*prior_multiplier).expand([X.shape[-1]]).to_event(1))
        eta = eta + X @ beta
    if model_id == "M2":
        tau_pair = numpyro.sample("tau_pair", dist.HalfNormal(PAIR_SD*prior_multiplier))
        z_pair = numpyro.sample("z_pair", dist.Normal(0, 1).expand([n_countries, n_countries]).to_event(2))
        pair = tau_pair*centered_pairs(z_pair)
        eta = eta + pair[voter_idx[:, None], recipient_idx]
    lp = categorical_log_probs(eta, mask)
    numpyro.factor("national_votes", lp[jnp.arange(len(y)), y].sum())


def predictive_eta(samples, packed, model_id, prior_multiplier=1.0):
    z = samples["z"]
    sigma = (samples["tau_country"] if model_id == "M2"
             else np.full(z.shape[:2], COUNTRY_SD*prior_multiplier))
    a = sigma[..., None]*(z-z.mean(axis=-1, keepdims=True))
    eta = a[..., packed["recipient_idx"]]
    if model_id != "M0":
        eta = eta + np.einsum("cdp,gkp->cdgk", samples["beta"], packed["X"])
    if model_id == "M2":
        raw_pair = samples["z_pair"]
        voters, recipients = packed["voter_idx"], packed["recipient_idx"]
        pair = (raw_pair[..., voters[:, None], recipients]
                -raw_pair.mean(axis=-1)[..., voters, None]
                -raw_pair.mean(axis=-2)[..., recipients]
                +raw_pair.mean(axis=(-2, -1))[..., None, None])
        eta = eta + samples["tau_pair"][..., None, None]*pair
    return eta


## 4. Обучение и сходимость

Как в HA4: NumPyro/NUTS, четыре цепи, нецентрированные эффекты.
Требуем R-hat ≤ 1.01, bulk ESS ≥ 400, tail ESS ≥ 200 и отсутствие divergences.
Неудачный расчёт один раз повторяем с большей длиной и точностью.
Проверки сходимости не используют качество на 2025. Промежуточные posterior сохраняются.


In [ ]:
def fit_posterior(fold, model_id, seed, prior_multiplier=1.0):
    settings = dict(model=model_id, prior_multiplier=prior_multiplier,
                    features=FEATURES, countries=fold["countries"], seed=int(seed),
                    country_sd=COUNTRY_SD, beta_sd=BETA_SD, tau_sd=TAU_SD, pair_sd=PAIR_SD,
                    chains=NUM_CHAINS, warmup=NUM_WARMUP, samples=NUM_SAMPLES,
                    model_version=MODEL_VERSION, numpyro=numpyro.__version__, jax=jax.__version__)
    signature = hashlib.sha256(json.dumps(settings, sort_keys=True).encode())
    signature.update(choice_model.__code__.co_code)
    signature.update(centered_pairs.__code__.co_code)
    signature.update(categorical_log_probs.__code__.co_code)
    signature.update(repr(choice_model.__code__.co_consts).encode())
    for key in ("X", "recipient_idx", "voter_idx", "mask", "y"):
        signature.update(np.ascontiguousarray(fold[key]).tobytes())
    stem = f"{model_id}_{fold['test'].year.iloc[0]}_{signature.hexdigest()[:16]}"
    cache = OUTPUT_DIR / "posterior"
    cache.mkdir(exist_ok=True)
    npz, meta = cache / f"{stem}.npz", cache / f"{stem}.json"
    if npz.exists() and meta.exists():
        report = json.loads(meta.read_text())
        if report["passed"]:
            report["prior_multiplier"] = float(prior_multiplier)
            print(f"  {model_id}: проверенный posterior из сохранённого расчёта", flush=True)
            with np.load(npz, allow_pickle=False) as loaded:
                return {k: loaded[k] for k in loaded.files}, report
    for attempt in range(2):
        print(f"  {model_id}: NUTS, попытка {attempt + 1}", flush=True)
        kernel = NUTS(choice_model, target_accept_prob=0.95 if attempt == 0 else 0.99,
                      max_tree_depth=10 if attempt == 0 else 12)
        mcmc = MCMC(kernel, num_warmup=NUM_WARMUP * (attempt + 1),
                    num_samples=NUM_SAMPLES * (attempt + 1), num_chains=NUM_CHAINS,
                    chain_method="sequential", progress_bar=True)
        started = time.perf_counter()
        mcmc.run(jax.random.PRNGKey(seed + attempt),
                 jnp.asarray(fold["X"]), jnp.asarray(fold["recipient_idx"]),
                 jnp.asarray(fold["voter_idx"]), jnp.asarray(fold["mask"]), jnp.asarray(fold["y"]), len(fold["countries"]), model_id,
                 prior_multiplier, extra_fields=("diverging", "num_steps"))
        samples = {k: np.asarray(v) for k, v in mcmc.get_samples(group_by_chain=True).items()}
        extra = mcmc.get_extra_fields(group_by_chain=True)
        diag = az.summary(az.from_dict(posterior=samples), kind="diagnostics", round_to=6)
        diag.to_csv(cache / f"{stem}_diagnostics.csv")
        depth_limit = 2 ** (10 if attempt == 0 else 12) - 1
        report = dict(model=model_id, year=int(fold["test"].year.iloc[0]),
                      prior_multiplier=float(prior_multiplier),
                      train_start=int(fold["train"].year.min()),
                      train_end=int(fold["train"].year.max()),
                      rhat_max=float(diag.r_hat.max()),
                      ess_bulk_min=float(diag.ess_bulk.min()),
                      ess_tail_min=float(diag.ess_tail.min()),
                      divergences=int(np.asarray(extra["diverging"]).sum()),
                      max_depth_hits=int((np.asarray(extra["num_steps"]) >= depth_limit).sum()),
                      seconds=time.perf_counter() - started, attempt=attempt + 1)
        report["passed"] = bool(
            np.isfinite(diag[["r_hat", "ess_bulk", "ess_tail"]]).all().all()
            and report["rhat_max"] <= 1.01 and report["ess_bulk_min"] >= 400
            and report["ess_tail_min"] >= 200 and report["divergences"] == 0
            and report["max_depth_hits"] == 0)
        meta.write_text(json.dumps(report, indent=2), encoding="utf-8")
        print(" ", report, flush=True)
        if report["passed"]:
            np.savez_compressed(npz, **samples)
            return samples, report
    raise RuntimeError(f"{model_id}: диагностика не пройдена. См. {cache / stem}")


def forecast(samples, fold, model_id, prior_multiplier=1.0):
    packed = fold["test_packed"]
    eta = predictive_eta(samples, packed, model_id, prior_multiplier)
    safe = np.where(packed["mask"], eta, -1e30)
    lp = safe-logsumexp(safe, axis=-1, keepdims=True)
    valid_lp = lp[..., packed["mask"]]
    p_draw = np.exp(valid_lp)
    result = fold["test"][["year", "voter", "recipient", "candidate_count"]].copy()
    result["model"] = model_id
    result["p"] = p_draw.mean(axis=(0, 1))
    result["log_p"] = logsumexp(valid_lp, axis=(0, 1))-np.log(valid_lp.shape[0]*valid_lp.shape[1])
    result["p_q05"], result["p_q95"] = np.quantile(p_draw.reshape(-1, p_draw.shape[-1]), [0.05, 0.95], axis=0)
    result["mcse_p"] = az.mcse(az.from_dict(posterior={"p": p_draw}), method="mean").p.values
    result["train_end"] = int(fold["train"].year.max())
    assert np.isfinite(result.select_dtypes("number")).all().all()
    assert np.allclose(result.groupby(["year", "voter"]).p.sum(), 1)
    assert not result.voter.eq(result.recipient).any()
    if result.mcse_p.max() > 0.01:
        warnings.warn("MCSE вероятности выше 0.01; для большей точности увеличьте NUM_SAMPLES.")
    return result


def score_forecast(prediction, truth):
    joined = prediction.merge(truth, on=["year", "voter", "recipient"], validate="one_to_one")
    assert len(joined) == len(prediction)
    rows = []
    for (year, voter), g in joined.groupby(["year", "voter"]):
        assert g[TARGET].sum() == 1 and np.isclose(g.p.sum(), 1)
        winner = g.loc[g[TARGET].eq(1)].iloc[0]
        favorite = g.loc[g.p.idxmax()]
        rows.append(dict(year=int(year), voter=voter, model=g.model.iloc[0],
                         log_loss=float(-winner.log_p), brier=float(((g.p-g[TARGET])**2).sum()),
                         accuracy=float(favorite.recipient == winner.recipient),
                         top3_accuracy=float(winner.recipient in set(g.nlargest(3, "p").recipient)),
                         actual_winner=winner.recipient, predicted_winner=favorite.recipient,
                         p_actual_winner=float(winner.p), confidence=float(favorite.p)))
    groups = pd.DataFrame(rows)
    metrics = groups[["log_loss", "brier", "accuracy", "top3_accuracy"]].mean().to_dict()
    return dict(model=prediction.model.iloc[0], year=int(prediction.year.iloc[0]),
                n_votes=len(groups), **metrics), groups


def run_year(model_id, train, design, seed, prior_multiplier=1.0):
    fold = prepare_fold(train, design)
    samples, diagnostic = fit_posterior(fold, model_id, seed, prior_multiplier)
    return dict(prediction=forecast(samples, fold, model_id, prior_multiplier),
                samples=samples, fold=fold, diagnostic=diagnostic)


## 5. Проверки реализации и prior predictive

Проверяем маскирование, сдвиговую инвариантность softmax, двойное центрирование
направленных пар и стандартизацию только по train.
До обучения смотрим, насколько уверенные прогнозы задают priors.


In [ ]:
test_eta = jnp.array([[0., 1., 1000.]])
test_mask = jnp.array([[True, True, False]])
lp = np.asarray(categorical_log_probs(test_eta, test_mask))
assert np.isclose(np.exp(lp).sum(), 1) and np.exp(lp[0, 2]) == 0
assert np.allclose(lp[:, :2], np.asarray(categorical_log_probs(test_eta+50, test_mask))[:, :2])
assert np.isfinite(np.asarray(jax.grad(lambda e: categorical_log_probs(e, test_mask)[0, 0])(test_eta))).all()
matrix = np.random.default_rng(SEED).normal(size=(8, 8))
centered = centered_pairs(matrix)
assert np.allclose(centered.mean(axis=0), 0) and np.allclose(centered.mean(axis=1), 0)
assert not np.allclose(centered, centered.T)
check = prepare_fold(development, holdout_design)
changed = holdout_design.copy()
changed["bpm"] += 10_000
assert np.array_equal(check["X"], prepare_fold(development, changed)["X"])
assert check["train"].year.max() == 2024 and TARGET not in check["test"]
assert check["mean"].loc[list(BINARY_FEATURES)].eq(0).all()
first_year = DEV_YEARS[0]
prior_fold = prepare_fold(development.loc[development.year.lt(first_year)],
                         development.loc[development.year.eq(first_year), holdout_design.columns])
prior_rows = []
for i, model_id in enumerate(MODELS):
    rng = np.random.default_rng(SEED+i)
    s = {"z": rng.normal(size=(1, 500, len(COUNTRIES)))}
    if model_id != "M0":
        s["beta"] = rng.normal(0, BETA_SD, size=(1, 500, len(FEATURES)))
    if model_id == "M2":
        s.update(tau_country=np.abs(rng.normal(0, TAU_SD, size=(1, 500))),
                 tau_pair=np.abs(rng.normal(0, PAIR_SD, size=(1, 500))),
                 z_pair=rng.normal(size=(1, 500, len(COUNTRIES), len(COUNTRIES))))
    eta = predictive_eta(s, prior_fold["test_packed"], model_id)
    p = softmax(np.where(prior_fold["test_packed"]["mask"], eta, -1e30), axis=-1)
    maximum = p.max(axis=-1)
    prior_rows.append(dict(model=model_id, median_max_probability=np.median(maximum),
                           q95_max_probability=np.quantile(maximum, 0.95),
                           share_max_above_95=np.mean(maximum > 0.95)))
prior_predictive = pd.DataFrame(prior_rows)
display(prior_predictive.round(4))
print("Проверки likelihood, центрирования, состава кандидатов и train/test: OK")


## 6. Прогнозы 2021–2024

Для каждого года обучаемся на всех предыдущих конкурсах, начиная с 2016.
Все модели используют одинаковые полные голосования.
Категориальный log loss и Brier: меньше — лучше. Brier суммирует квадраты ошибок
по кандидатам, затем усредняется по голосованиям. Каждый конкурс имеет одинаковый вес.


In [ ]:
development_runs, development_scores, development_groups, diagnostics = {}, [], [], []
for year in DEV_YEARS:
    train = development.loc[development.year.lt(year)]
    design = development.loc[development.year.eq(year), holdout_design.columns]
    truth = development.loc[development.year.eq(year), holdout_truth.columns]
    print(f"\nПроверка {year}: train {train.year.min()}–{train.year.max()}", flush=True)
    for i, model_id in enumerate(MODELS):
        run = run_year(model_id, train, design, SEED+year*10+i)
        development_runs[(year, model_id)] = run
        score, groups = score_forecast(run["prediction"], truth)
        development_scores.append(score)
        development_groups.append(groups)
        diagnostics.append(run["diagnostic"])
dev_metrics = pd.DataFrame(development_scores)
dev_summary = dev_metrics.groupby("model")[["log_loss", "brier", "accuracy", "top3_accuracy"]].mean()
display(dev_summary.round(4))


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for model_id in MODELS:
    g = dev_metrics.loc[dev_metrics.model.eq(model_id)]
    for ax, metric in zip(axes, ["log_loss", "brier"]):
        ax.plot(g.year, g[metric], "o-", color=COLORS[model_id], label=model_id)
        ax.xaxis.set_major_locator(MaxNLocator(integer=True))
        ax.grid(axis="y", color="#E5EAF1")
axes[0].set(title="Категориальный log loss", xlabel="Тестовый год", ylabel="Меньше — лучше")
axes[1].set(title="Multiclass Brier", xlabel="Тестовый год", ylabel="Меньше — лучше")
axes[0].legend(frameon=False)
fig.suptitle("Национальные голосования: проверка до 2025", fontsize=14)
fig.savefig(OUTPUT_DIR / "development_scores.png", dpi=200, bbox_inches="tight")
plt.show()
paired_rows = []
for metric in ["log_loss", "brier"]:
    pivot = dev_metrics.pivot(index="year", columns="model", values=metric)
    for newer, older in [("M1", "M0"), ("M2", "M1")]:
        delta = (pivot[newer]-pivot[older]).to_numpy()
        boot = np.random.default_rng(SEED).choice(delta, size=(10_000, len(delta)), replace=True).mean(axis=1)
        lo, hi = np.quantile(boot, [0.025, 0.975])
        paired_rows.append(dict(comparison=f"{newer} − {older}", metric=metric,
                                mean_difference=delta.mean(), q025=lo, q975=hi, n_years=len(delta)))
paired_comparisons = pd.DataFrame(paired_rows)
display(paired_comparisons.round(4))
print("Отрицательная разница — улучшение; интервалы по четырём годам ориентировочные.")


## 7. Чувствительность priors

Только на 2024 повторяем расчёт с удвоенными масштабами priors.
Основные модели и прогноз 2025 сохраняют исходные настройки.
Проверку можно отключить через RUN_SENSITIVITY; это не меняет основные модели.


In [ ]:
sensitivity_rows = []
if RUN_SENSITIVITY:
    year = DEV_YEARS[-1]
    train = development.loc[development.year.lt(year)]
    design = development.loc[development.year.eq(year), holdout_design.columns]
    truth = development.loc[development.year.eq(year), holdout_truth.columns]
    for i, model_id in enumerate(MODELS):
        run = run_year(model_id, train, design, SEED+year*10+i, prior_multiplier=2.0)
        score, _ = score_forecast(run["prediction"], truth)
        sensitivity_rows.append(score | {"prior_scale": "2x"})
        diagnostics.append(run["diagnostic"])
    sensitivity = pd.concat([dev_metrics.loc[dev_metrics.year.eq(year)].assign(prior_scale="1x"),
                             pd.DataFrame(sensitivity_rows)], ignore_index=True)
    display(sensitivity[["model", "prior_scale", "log_loss", "brier"]].round(4))


## 8. Финальный тест: 2025

Обучаемся на 2016–2024. После выпуска вероятностей присоединяем исходы 2025.
Для каждого голосующего показываем реального и прогнозных победителей,
а также вероятность, присвоенную реальному победителю.


In [ ]:
holdout_runs, holdout_scores, holdout_groups = {}, [], []
for i, model_id in enumerate(MODELS):
    print(f"\nФинальный тест 2025: {model_id}", flush=True)
    run = run_year(model_id, development, holdout_design, SEED+HOLDOUT_YEAR*10+i)
    holdout_runs[model_id] = run
    score, groups = score_forecast(run["prediction"], holdout_truth)
    holdout_scores.append(score)
    holdout_groups.append(groups)
    diagnostics.append(run["diagnostic"])
holdout_metrics = pd.DataFrame(holdout_scores).set_index("model")
predictions_2025 = pd.concat([r["prediction"] for r in holdout_runs.values()], ignore_index=True)
comparison_2025 = pd.concat(holdout_groups, ignore_index=True)
display(holdout_metrics.round(4))
table_2025 = comparison_2025.pivot(index="voter", columns="model", values=["predicted_winner", "p_actual_winner"])
table_2025.columns = [f"{model}_{metric}" for metric, model in table_2025.columns]
table_2025 = table_2025.join(comparison_2025.groupby("voter").actual_winner.first())
display(table_2025.style.format({c: "{:.1%}" for c in table_2025 if "p_actual" in c}))


## 9. Карта вероятностей M2

Строки — голосующие страны, столбцы — финалисты. Квадрат отмечает реального
национального победителя, белая точка — самого вероятного по M2.
Серые клетки — запрещённое самоголосование. Вероятности оценивают весь допустимый набор.


In [ ]:
model2 = holdout_runs["M2"]["prediction"]
recipient_order = model2.groupby("recipient").p.sum().sort_values(ascending=False).index.tolist()
actual_winners = holdout_truth.loc[holdout_truth[TARGET].eq(1)].set_index("voter").recipient
voter_order = sorted(actual_winners.index, key=lambda v: (recipient_order.index(actual_winners[v]), v))
heat = model2.pivot(index="voter", columns="recipient", values="p").reindex(index=voter_order, columns=recipient_order)
fig, ax = plt.subplots(figsize=(14, max(8, len(voter_order)*0.28)), layout="constrained")
cmap = plt.get_cmap("YlGnBu").copy()
cmap.set_bad("#E8ECF0")
image = ax.imshow(np.ma.masked_invalid(heat.to_numpy(float)), aspect="auto", interpolation="nearest",
                  cmap=cmap, vmin=0, vmax=max(0.2, np.ceil(np.nanmax(heat.to_numpy())*10)/10))
for row, voter in enumerate(voter_order):
    actual_col = recipient_order.index(actual_winners[voter])
    predicted_col = int(np.nanargmax(heat.loc[voter].to_numpy(float)))
    ax.scatter(actual_col, row, marker="s", s=60, facecolors="none", edgecolors="#202C39", linewidths=1.2)
    ax.scatter(predicted_col, row, marker="o", s=16, c="white", edgecolors="#65758A", linewidths=0.5)
ax.set_xticks(range(len(recipient_order)), recipient_order, fontsize=9)
ax.set_yticks(range(len(voter_order)), voter_order, fontsize=9)
ax.set(xlabel="Получатель", ylabel="Голосующая страна", title="2025: вероятность победителя национального televote · M2")
bar = fig.colorbar(image, ax=ax, fraction=0.03, pad=0.02)
bar.ax.yaxis.set_major_formatter(PercentFormatter(1, decimals=0))
ax.legend(handles=[Line2D([], [], marker="s", linestyle="none", markerfacecolor="none", markeredgecolor="#202C39", label="Фактический победитель"),
                    Line2D([], [], marker="o", linestyle="none", markerfacecolor="white", markeredgecolor="#65758A", label="Прогноз M2")],
          loc="upper left", bbox_to_anchor=(0, -0.07), ncol=2, frameon=False)
fig.savefig(OUTPUT_DIR / "probabilities_2025.png", dpi=220, bbox_inches="tight")
fig.savefig(OUTPUT_DIR / "probabilities_2025.pdf", bbox_inches="tight")
plt.show()


## 10. Ожидаемое и фактическое число первых мест

Суммируем вероятности по голосующим: получаем ожидаемое число национальных побед.
Это не сумма всех зрительских баллов и не итоговое место в Grand Final.


In [ ]:
actual_counts = actual_winners.value_counts().reindex(recipient_order, fill_value=0)
y = np.arange(len(recipient_order))
fig, ax = plt.subplots(figsize=(10, max(7, len(y)*0.29)), layout="constrained")
ax.barh(y, actual_counts, color="#D8DFE6", height=0.76, label="Фактическое число")
for i, model_id in enumerate(MODELS):
    expected = holdout_runs[model_id]["prediction"].groupby("recipient").p.sum().reindex(recipient_order)
    ax.scatter(expected, y+(i-1)*0.18, color=COLORS[model_id], label=model_id, s=28)
ax.set_yticks(y, recipient_order)
ax.set_ylim(len(y)-0.5, -0.5)
ax.set(xlabel="Число национальных первых мест", title="2025: ожидаемая поддержка и факт")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.grid(axis="x", color="#E5EAF1")
ax.legend(frameon=False)
fig.savefig(OUTPUT_DIR / "national_wins_2025.png", dpi=200, bbox_inches="tight")
plt.show()


## 11. Калибровка и диагностика

Для каждого голосования берём уверенность в самом вероятном кандидате и проверяем,
как часто он действительно побеждал. Используем только 2021–2024;
каждый конкурс имеет одинаковый суммарный вес. При четырёх годах график описательный.


In [ ]:
group_results = pd.concat(development_groups, ignore_index=True)
calibration_tables = []
fig, ax = plt.subplots(figsize=(6, 5), layout="constrained")
ax.plot([0, 1], [0, 1], "--", color="#A4AEB8", linewidth=1)
for model_id in MODELS:
    g = group_results.loc[group_results.model.eq(model_id)].copy()
    g["weight"] = 1/g.groupby("year").year.transform("size")
    g["bin"] = pd.cut(g.confidence, np.linspace(0, 1, 11), include_lowest=True, right=True)
    bins = pd.DataFrame([dict(model=model_id, bin=str(interval), n=len(b),
                              predicted=np.average(b.confidence, weights=b.weight),
                              observed=np.average(b.accuracy, weights=b.weight))
                         for interval, b in g.groupby("bin", observed=True)])
    calibration_tables.append(bins)
    ax.plot(bins.predicted, bins.observed, "o-", color=COLORS[model_id], label=model_id)
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="Вероятность самого вероятного кандидата",
       ylabel="Доля верных первых мест", title="Калибровка до 2025")
ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.yaxis.set_major_formatter(PercentFormatter(1))
ax.grid(color="#E5EAF1")
ax.legend(frameon=False)
fig.savefig(OUTPUT_DIR / "calibration_development.png", dpi=180, bbox_inches="tight")
plt.show()
display(pd.DataFrame([r["diagnostic"] for r in holdout_runs.values()]))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), layout="constrained")
for ax, parameter, title in zip(axes, ["tau_country", "tau_pair"], ["Масштаб получателей", "Масштаб пар"]):
    for chain in holdout_runs["M2"]["samples"][parameter]:
        ax.plot(chain, linewidth=0.45, alpha=0.65)
    ax.set(title=f"M2: {title}", xlabel="Сохранённая итерация")
fig.savefig(OUTPUT_DIR / "posterior_traces.png", dpi=180, bbox_inches="tight")
plt.show()


## 12. Сохранение

Сохраняем вероятности каждого допустимого получателя, результаты по голосованиям,
годовые метрики, диагностику, posterior draws и стандартизацию.
Интервалы вероятности отражают неопределённость параметров; MCSE — вычислительную погрешность.
Модель гарантирует одного победителя внутри голосования, но не полную независимость
ошибок разных голосующих стран. Коэффициенты не доказывают причинное влияние.


In [ ]:
pd.concat([r["prediction"] for r in development_runs.values()], ignore_index=True).to_csv(OUTPUT_DIR / "development_predictions.csv", index=False)
group_results.to_csv(OUTPUT_DIR / "development_group_metrics.csv", index=False)
dev_metrics.to_csv(OUTPUT_DIR / "development_metrics_by_year.csv", index=False)
dev_summary.to_csv(OUTPUT_DIR / "development_summary.csv")
paired_comparisons.to_csv(OUTPUT_DIR / "paired_comparisons.csv", index=False)
predictions_2025.to_csv(OUTPUT_DIR / "predictions_2025.csv", index=False)
comparison_2025.to_csv(OUTPUT_DIR / "comparison_2025.csv", index=False)
table_2025.to_csv(OUTPUT_DIR / "national_winners_2025.csv")
holdout_metrics.to_csv(OUTPUT_DIR / "metrics_2025.csv")
pd.DataFrame(diagnostics).to_csv(OUTPUT_DIR / "diagnostics.csv", index=False)
prior_predictive.to_csv(OUTPUT_DIR / "prior_predictive.csv", index=False)
pd.concat(calibration_tables).to_csv(OUTPUT_DIR / "calibration.csv", index=False)
if sensitivity_rows:
    sensitivity.to_csv(OUTPUT_DIR / "prior_sensitivity_2024.csv", index=False)
pd.DataFrame({"mean": holdout_runs["M1"]["fold"]["mean"], "scale": holdout_runs["M1"]["fold"]["scale"]}).to_csv(OUTPUT_DIR / "training_scaler_2025.csv")
feature_effects = []
for model_id in ["M1", "M2"]:
    b = holdout_runs[model_id]["samples"]["beta"].reshape(-1, len(FEATURES))
    for j, feature in enumerate(FEATURES):
        feature_effects.append(dict(model=model_id, feature=feature, mean=b[:, j].mean(),
                                    q05=np.quantile(b[:, j], 0.05), q95=np.quantile(b[:, j], 0.95)))
pd.DataFrame(feature_effects).to_csv(OUTPUT_DIR / "feature_effects_2025.csv", index=False)
manifest = dict(task="C", experiment=EXPERIMENT, sheet=SHEET, features=FEATURES,
                recipient_features=RECIPIENT_FEATURES, pair_features=PAIR_FEATURES,
                countries=COUNTRIES, holdout=HOLDOUT_YEAR, development_test_years=DEV_YEARS,
                train_rule="year < test_year", seed=SEED, model_version=MODEL_VERSION,
                country_sd=COUNTRY_SD, beta_sd=BETA_SD, tau_sd=TAU_SD, pair_sd=PAIR_SD,
                chains=NUM_CHAINS, warmup=NUM_WARMUP, samples=NUM_SAMPLES,
                input_sha256=hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
                versions=dict(numpyro=numpyro.__version__, jax=jax.__version__, arviz=az.__version__),
                market_snapshot_time="unknown_accepted" if EXPERIMENT=="market" else "not_used")
(OUTPUT_DIR / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
for newer, older in [("M1", "M0"), ("M2", "M1")]:
    pivot = dev_metrics.pivot(index="year", columns="model", values="log_loss")
    print(f"До 2025: log loss({newer}) − log loss({older}) = {(pivot[newer]-pivot[older]).mean():+.4f}.")
archive = shutil.make_archive(str(OUTPUT_DIR), "zip", OUTPUT_DIR)
print(f"Результаты: {OUTPUT_DIR.resolve()}\nАрхив: {archive}")


## Основа в материалах курса

HA3 — модели выбора, интегрирование вероятностей по posterior и устойчивый log score;
HA4 — NumPyro/NUTS и нецентрированная иерархия; S3 — собственный likelihood.
Multinomial logit и эффекты направленных пар — адаптация к нашей задаче,
а не готовая функция из ДЗ. Brier и калибровка опираются на HA1.
